# Notebook 03 — Baseline Models
## Adaptive Reliability-Aware Fusion (ARAF) Project

**Goal of this notebook:**  
Build three baseline models that ARAF will be compared against. These are
deliberately simpler than ARAF — their job is to establish a performance
floor and reveal exactly what naive approaches get wrong under corruption.

**Why baselines matter in research:**  
A model that achieves 70% accuracy sounds impressive. A model that achieves
70% accuracy when the best possible baseline achieves 68% is barely an
improvement. Baselines give your results meaning. Without them, you have
no claim.

**The three baselines we build:**

| Model | What it uses | What it ignores | Why it matters |
|---|---|---|---|
| `UnimodalImageModel` | Image only | Text entirely | Upper bound of vision-only approach |
| `UnimodalTextModel` | Text only | Image entirely | Upper bound of language-only approach |
| `NaiveFusionModel` | Image + Text | Reliability of each | Shows cost of ignoring corruption |

**What you will learn:**
- How pretrained encoders (ResNet, BERT) work as feature extractors
- What "freezing" vs "fine-tuning" means and when to do each
- How to build a classification head on top of encoder features
- What features look like in high-dimensional space (PCA visualization)
- How corruption degrades features — and why naive fusion fails

---


## 1. Imports and setup


In [ ]:
import os, sys, json, copy, random
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from transformers import BertModel, BertTokenizer
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Install scikit-learn if needed (for PCA visualization)
try:
    from sklearn.decomposition import PCA
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "scikit-learn", "-q"])
    from sklearn.decomposition import PCA

print("Imports successful.")


## 2. Reload shared components from previous notebooks

We reload `MultimodalSample`, the dataset, and the `CorruptionModule`.
In a future refactor these will all be proper imports from `.py` files.


In [ ]:
# ── MultimodalSample (same as Notebooks 01 and 02) ───────────────────────────
@dataclass
class MultimodalSample:
    image: torch.Tensor
    text_ids: torch.Tensor
    attention_mask: torch.Tensor
    label: torch.Tensor
    raw_image: Optional[object] = None
    raw_text: str = ""
    dataset_name: str = "vqa_v2"
    sample_id: str = ""
    image_corrupted: bool = False
    text_corrupted: bool = False
    image_missing: bool = False
    text_missing: bool = False
    corruption_severity: float = 0.0

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMAGE_SIZE    = 224
MAX_TEXT_LEN  = 32

clean_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img  = (tensor * std + mean).clamp(0, 1)
    return (img * 255).byte().permute(1, 2, 0).numpy()

print("Shared components defined.")


In [ ]:
# ── Load corruption module from the .py file we saved in Notebook 02 ─────────
import importlib.util
spec = importlib.util.spec_from_file_location(
    "corruption_module", "corruption/corruption_module.py")
corruption_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(corruption_mod)
CorruptionModule = corruption_mod.CorruptionModule
print("CorruptionModule loaded from corruption/corruption_module.py")

# ── Load a small dataset for testing models ───────────────────────────────────
from datasets import load_dataset
from collections import Counter
from transformers import BertTokenizer

print("Loading VQA v2 samples...")
hf_val_full = load_dataset("lmms-lab/VQAv2", split="validation",
                           trust_remote_code=True)
with open("answer_vocab.json") as f:
    answer2idx = json.load(f)
idx2answer = {v: k for k, v in answer2idx.items()}
NUM_CLASSES = len(answer2idx)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def load_sample(row):
    pil = row["image"].convert("RGB")
    img = clean_transform(pil)
    enc = tokenizer(row["question"], padding="max_length",
                    max_length=MAX_TEXT_LEN, truncation=True,
                    return_tensors="pt")
    label = torch.zeros(NUM_CLASSES)
    cnt = Counter(a["answer"].lower().strip() for a in row["answers"])
    for ans, c in cnt.items():
        if ans in answer2idx:
            label[answer2idx[ans]] = min(c / 3.0, 1.0)
    return MultimodalSample(
        image=img, text_ids=enc["input_ids"].squeeze(0),
        attention_mask=enc["attention_mask"].squeeze(0),
        label=label, raw_image=pil, raw_text=row["question"],
        dataset_name="vqa_v2",
        sample_id=str(row.get("question_id", 0)),
    )

# Load 64 samples — enough to visualize features meaningfully
print("Loading 64 samples...")
samples = [load_sample(hf_val_full[i]) for i in range(64)]
print(f"Loaded {len(samples)} clean samples.")


## 3. What is an encoder? (concept before code)

Before writing any model code, it's important to understand what an encoder
does conceptually.

### The core idea

An encoder takes raw input (pixels or tokens) and maps it to a
**dense feature vector** — a compact numerical representation that captures
the semantic meaning of the input.

```
Raw image  [3, 224, 224]  →  Image encoder  →  Feature vector [2048]
Raw text   [32 tokens]    →  Text encoder   →  Feature vector [768]
```

The feature vector for "a red bus" should be similar to "a crimson bus" and
different from "a blue bicycle." The encoder learns this during pretraining
on massive datasets.

### Why use pretrained encoders?

Training an image encoder from scratch on VQA v2 would require millions of
images and weeks of compute. ResNet-50 was already trained on 1.2 million
ImageNet images. BERT was trained on the entire English Wikipedia + BookCorpus.

We get their learned representations for free and just train a small
classification head on top. This is called **transfer learning** and is
standard practice in all modern deep learning research.

### Frozen vs fine-tuned

- **Frozen**: encoder weights don't change during training. Fast, uses less
  memory, works well when your dataset is small.
- **Fine-tuned**: encoder weights update slowly during training. Better
  performance but needs more data and careful learning rate scheduling.

For our baselines we freeze the encoders. For ARAF (Notebook 04) we will
optionally fine-tune the last few layers.


## 4. Image encoder — ResNet-50

ResNet-50 is a 50-layer convolutional neural network pretrained on ImageNet.
We use it as a feature extractor by removing its final classification layer
and keeping everything up to the global average pooling layer.

### What ResNet-50 outputs

The last layer before classification produces a `[2048]` dimensional feature
vector for each image. This vector encodes things like: what objects are
present, what colors, what spatial relationships.

### Why ResNet-50 specifically?

It's the standard baseline encoder in multimodal research. It's large enough
to be expressive but small enough to run on a laptop GPU (or even CPU for
small batches). Later you could swap in a ViT (Vision Transformer) for better
performance — our code is designed to make this easy.


In [ ]:
class ImageEncoder(nn.Module):
    """
    ResNet-50 feature extractor.
    Removes the classification head, outputs [batch, 2048] feature vectors.
    
    Args:
        frozen: if True, all ResNet weights are frozen (no gradient updates).
                Set to False to fine-tune the encoder.
        output_dim: if not None, adds a linear projection to this dimension.
                    Useful for matching text encoder dimension (768).
    """
    def __init__(self, frozen: bool = True, output_dim: Optional[int] = None):
        super().__init__()
        
        # Load pretrained ResNet-50
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        
        # Remove the final FC layer — we want features, not ImageNet classes
        # resnet.fc maps [2048] -> [1000 ImageNet classes]
        # We replace it with Identity() so features pass through unchanged
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        # Output shape after this: [batch, 2048, 1, 1]
        
        self.feature_dim = 2048
        
        # Optional projection to a different dimension
        self.projection = None
        if output_dim is not None:
            self.projection = nn.Sequential(
                nn.Linear(2048, output_dim),
                nn.LayerNorm(output_dim),
                nn.ReLU(),
            )
            self.feature_dim = output_dim
        
        # Freeze all encoder weights if requested
        if frozen:
            for param in self.encoder.parameters():
                param.requires_grad = False
            print("ImageEncoder: ResNet-50 weights FROZEN")
        else:
            print("ImageEncoder: ResNet-50 weights TRAINABLE")
    
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: [B, 3, 224, 224] normalized float32
        Returns:
            features: [B, feature_dim]
        """
        # ResNet forward pass
        features = self.encoder(images)          # [B, 2048, 1, 1]
        features = features.flatten(start_dim=1) # [B, 2048]
        
        if self.projection is not None:
            features = self.projection(features) # [B, output_dim]
        
        return features


# ── Test the image encoder ────────────────────────────────────────────────────
print("Building ImageEncoder...")
image_encoder = ImageEncoder(frozen=True, output_dim=None)
image_encoder = image_encoder.to(DEVICE)

# Forward pass with a batch of 4 images
test_images = torch.stack([samples[i].image for i in range(4)]).to(DEVICE)
with torch.no_grad():
    img_features = image_encoder(test_images)

print(f"Input shape  : {test_images.shape}")
print(f"Output shape : {img_features.shape}   # [batch=4, features=2048]")
print(f"Feature range: [{img_features.min():.3f}, {img_features.max():.3f}]")
print(f"Feature mean : {img_features.mean():.3f}")

# Count trainable parameters
total_params    = sum(p.numel() for p in image_encoder.parameters())
trainable_params = sum(p.numel() for p in image_encoder.parameters()
                       if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}  (0 because frozen)")


## 5. Text encoder — BERT

BERT (Bidirectional Encoder Representations from Transformers) processes a
sequence of tokens and produces a contextual representation for each token.
We use the `[CLS]` token's representation as the sentence-level feature vector.

### Why the `[CLS]` token?

BERT was designed so that the `[CLS]` token (always the first token) aggregates
information from the entire sequence through the attention mechanism. During
BERT's pretraining, the `[CLS]` representation was used for classification
tasks — so it's the most semantically rich single vector we can extract.

### BERT output dimension

BERT-base outputs `[768]` dimensional vectors. This is smaller than ResNet's
`[2048]`. In our fusion model we will project both to the same dimension
before combining them.


In [ ]:
class TextEncoder(nn.Module):
    """
    BERT feature extractor.
    Uses the [CLS] token representation as the sentence feature vector.
    Outputs [batch, 768] feature vectors.
    
    Args:
        frozen    : if True, all BERT weights are frozen
        output_dim: if not None, adds a linear projection to this dimension
    """
    def __init__(self, frozen: bool = True, output_dim: Optional[int] = None):
        super().__init__()
        
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.feature_dim = 768  # BERT-base hidden size
        
        # Optional projection
        self.projection = None
        if output_dim is not None:
            self.projection = nn.Sequential(
                nn.Linear(768, output_dim),
                nn.LayerNorm(output_dim),
                nn.ReLU(),
            )
            self.feature_dim = output_dim
        
        if frozen:
            for param in self.bert.parameters():
                param.requires_grad = False
            print("TextEncoder: BERT weights FROZEN")
        else:
            print("TextEncoder: BERT weights TRAINABLE")
    
    def forward(self, text_ids: torch.Tensor,
                attention_mask: torch.Tensor) -> torch.Tensor:
        """
        Args:
            text_ids      : [B, seq_len] int64
            attention_mask: [B, seq_len] int64
        Returns:
            features: [B, feature_dim]  — the [CLS] token representation
        """
        outputs = self.bert(
            input_ids      = text_ids,
            attention_mask = attention_mask,
        )
        # outputs.last_hidden_state: [B, seq_len, 768]
        # Index 0 = [CLS] token — our sentence representation
        cls_features = outputs.last_hidden_state[:, 0, :]  # [B, 768]
        
        if self.projection is not None:
            cls_features = self.projection(cls_features)
        
        return cls_features


# ── Test the text encoder ─────────────────────────────────────────────────────
print("Building TextEncoder...")
text_encoder = TextEncoder(frozen=True, output_dim=None)
text_encoder = text_encoder.to(DEVICE)

test_ids   = torch.stack([samples[i].text_ids for i in range(4)]).to(DEVICE)
test_masks = torch.stack([samples[i].attention_mask for i in range(4)]).to(DEVICE)

with torch.no_grad():
    txt_features = text_encoder(test_ids, test_masks)

print(f"Input text_ids shape  : {test_ids.shape}")
print(f"Output feature shape  : {txt_features.shape}   # [batch=4, features=768]")
print(f"Feature range         : [{txt_features.min():.3f}, {txt_features.max():.3f}]")

total_params    = sum(p.numel() for p in text_encoder.parameters())
trainable_params = sum(p.numel() for p in text_encoder.parameters()
                        if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}  (0 because frozen)")


## 6. Classification head

The classification head sits on top of the encoder and maps feature vectors
to class logits (raw scores before softmax). It's the only part that trains
from scratch — the encoder is pretrained and frozen.

### Architecture: MLP with dropout

```
features [input_dim]
    → Linear(input_dim, hidden_dim)
    → ReLU
    → Dropout(0.3)
    → Linear(hidden_dim, num_classes)
    → [num_classes] logits
```

Dropout during training randomly zeros out 30% of neurons. This prevents
the classifier from memorizing the training set (overfitting) and forces it
to learn more robust representations.

### Why not softmax here?

We output raw logits, not probabilities. Our loss function (BCEWithLogitsLoss)
applies sigmoid internally and is numerically more stable than doing softmax
then cross-entropy separately.


In [ ]:
class ClassificationHead(nn.Module):
    """
    MLP classification head. Sits on top of any encoder.
    Takes feature vectors and outputs class logits.
    
    Args:
        input_dim  : dimension of input features (2048 for ResNet, 768 for BERT)
        hidden_dim : hidden layer size
        num_classes: number of output classes (3129 for VQA v2)
        dropout    : dropout rate during training
    """
    def __init__(self, input_dim: int, hidden_dim: int = 1024,
                 num_classes: int = 3129, dropout: float = 0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )
    
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            features: [B, input_dim]
        Returns:
            logits: [B, num_classes]  — raw scores, NOT probabilities
        """
        return self.classifier(features)


# ── Test ──────────────────────────────────────────────────────────────────────
head = ClassificationHead(input_dim=2048, num_classes=NUM_CLASSES).to(DEVICE)
test_logits = head(img_features)
print(f"Classifier input  : {img_features.shape}")
print(f"Classifier output : {test_logits.shape}   # [batch=4, num_classes=3129]")

trainable = sum(p.numel() for p in head.parameters() if p.requires_grad)
print(f"Trainable params  : {trainable:,}  (these train from scratch)")


## 7. Baseline 1 & 2 — Unimodal models

A unimodal model uses only one modality. It cannot adapt to missing or
corrupted inputs in the other modality because it never sees them.

### Why build unimodal baselines?

They answer two critical research questions:
1. How much does each modality contribute individually?
2. Does our fusion model actually do better than the best single modality?

If ARAF doesn't beat the best unimodal model, fusion isn't helping.


In [ ]:
class UnimodalImageModel(nn.Module):
    """
    Image-only baseline. Encodes image with ResNet-50, classifies with MLP.
    Completely ignores text input.
    
    Architecture:
        Image [3,224,224] -> ResNet-50 -> [2048] -> MLP -> [num_classes]
    """
    def __init__(self, num_classes: int = 3129, frozen_encoder: bool = True):
        super().__init__()
        self.encoder    = ImageEncoder(frozen=frozen_encoder)
        self.classifier = ClassificationHead(
            input_dim=self.encoder.feature_dim,
            num_classes=num_classes,
        )
        self.name = "UnimodalImage"
    
    def forward(self, batch: Dict) -> Dict:
        """
        Args:
            batch: dict from collate_multimodal (must have 'image' key)
        Returns:
            dict with 'logits' [B, num_classes] and 'features' [B, 2048]
        """
        images   = batch["image"].to(DEVICE)
        features = self.encoder(images)           # [B, 2048]
        logits   = self.classifier(features)      # [B, num_classes]
        return {"logits": logits, "features": features}


class UnimodalTextModel(nn.Module):
    """
    Text-only baseline. Encodes question with BERT, classifies with MLP.
    Completely ignores image input.
    
    Architecture:
        Text [32] -> BERT -> [CLS]=>[768] -> MLP -> [num_classes]
    """
    def __init__(self, num_classes: int = 3129, frozen_encoder: bool = True):
        super().__init__()
        self.encoder    = TextEncoder(frozen=frozen_encoder)
        self.classifier = ClassificationHead(
            input_dim=self.encoder.feature_dim,
            num_classes=num_classes,
        )
        self.name = "UnimodalText"
    
    def forward(self, batch: Dict) -> Dict:
        """
        Args:
            batch: dict from collate_multimodal
        Returns:
            dict with 'logits' [B, num_classes] and 'features' [B, 768]
        """
        text_ids = batch["text_ids"].to(DEVICE)
        attn     = batch["attention_mask"].to(DEVICE)
        features = self.encoder(text_ids, attn)   # [B, 768]
        logits   = self.classifier(features)      # [B, num_classes]
        return {"logits": logits, "features": features}


# ── Build and test both unimodal models ───────────────────────────────────────
print("Building unimodal models...")
img_model = UnimodalImageModel(num_classes=NUM_CLASSES).to(DEVICE)
txt_model = UnimodalTextModel(num_classes=NUM_CLASSES).to(DEVICE)

# Minimal batch for testing
test_batch = {
    "image"          : torch.stack([samples[i].image for i in range(4)]),
    "text_ids"       : torch.stack([samples[i].text_ids for i in range(4)]),
    "attention_mask" : torch.stack([samples[i].attention_mask for i in range(4)]),
    "label"          : torch.stack([samples[i].label for i in range(4)]),
}

with torch.no_grad():
    img_out = img_model(test_batch)
    txt_out = txt_model(test_batch)

print(f"Image model logits shape : {img_out['logits'].shape}")
print(f"Text  model logits shape : {txt_out['logits'].shape}")
print(f"Image model features     : {img_out['features'].shape}")
print(f"Text  model features     : {txt_out['features'].shape}")


## 8. Baseline 3 — Naive fusion model

The naive fusion model combines image and text features by concatenating them
and passing through an MLP. It has no concept of reliability — it treats both
modalities as equally trustworthy at all times.

### Fusion by concatenation

```
Image features [2048]  ─┐
                         ├─ concat → [2816] → MLP → [num_classes]
Text features  [768]   ─┘
```

2048 + 768 = 2816 total features fed into the classifier.

### Why concatenation is naive

If the image is completely corrupted (zero tensor), the model still receives
the same 2048-dimensional input from the image encoder — but now those 2048
values are meaningless noise. The model has no way to know this and will
weight the noisy image features just as heavily as clean ones.

This is precisely the problem ARAF solves: instead of blind concatenation,
ARAF estimates how reliable each modality is and weights accordingly.

### What we expect to see

Under clean inputs: naive fusion should outperform both unimodal models
(it has more information).
Under corruption: naive fusion should degrade more than ARAF (it can't
discount the corrupted modality).

This gap between clean and corrupted performance is our key research finding.


In [ ]:
class NaiveFusionModel(nn.Module):
    """
    Concatenation-based fusion baseline.
    Combines image and text features with no reliability weighting.
    
    Architecture:
        Image [3,224,224] -> ResNet-50 -> [2048] ─┐
                                                   ├─ cat -> [2816] -> MLP -> [num_classes]
        Text  [32]        -> BERT     -> [768]  ─┘
    
    This is the standard baseline in multimodal papers.
    It represents the best you can do WITHOUT reliability awareness.
    """
    def __init__(self, num_classes: int = 3129, frozen_encoders: bool = True):
        super().__init__()
        self.image_encoder = ImageEncoder(frozen=frozen_encoders)
        self.text_encoder  = TextEncoder(frozen=frozen_encoders)
        
        # Combined feature dimension
        fusion_dim = self.image_encoder.feature_dim + self.text_encoder.feature_dim
        # 2048 + 768 = 2816
        
        self.classifier = ClassificationHead(
            input_dim  = fusion_dim,
            hidden_dim = 1024,
            num_classes= num_classes,
        )
        self.name = "NaiveFusion"
        
        print(f"NaiveFusionModel: image_dim={self.image_encoder.feature_dim}, "
              f"text_dim={self.text_encoder.feature_dim}, "
              f"fusion_dim={fusion_dim}")
    
    def forward(self, batch: Dict) -> Dict:
        """
        Args:
            batch: dict with 'image', 'text_ids', 'attention_mask'
        Returns:
            dict with 'logits', 'image_features', 'text_features', 'fused_features'
        """
        images   = batch["image"].to(DEVICE)
        text_ids = batch["text_ids"].to(DEVICE)
        attn     = batch["attention_mask"].to(DEVICE)
        
        # Encode each modality independently
        img_feat = self.image_encoder(images)          # [B, 2048]
        txt_feat = self.text_encoder(text_ids, attn)   # [B, 768]
        
        # Naive fusion: just concatenate
        fused    = torch.cat([img_feat, txt_feat], dim=1)  # [B, 2816]
        logits   = self.classifier(fused)                   # [B, num_classes]
        
        return {
            "logits"          : logits,
            "image_features"  : img_feat,
            "text_features"   : txt_feat,
            "fused_features"  : fused,
        }


# ── Build and test ────────────────────────────────────────────────────────────
print("Building NaiveFusionModel...")
fusion_model = NaiveFusionModel(num_classes=NUM_CLASSES).to(DEVICE)

with torch.no_grad():
    fused_out = fusion_model(test_batch)

print(f"Fusion model logits shape         : {fused_out['logits'].shape}")
print(f"Fusion model image features shape : {fused_out['image_features'].shape}")
print(f"Fusion model text features shape  : {fused_out['text_features'].shape}")
print(f"Fusion model fused features shape : {fused_out['fused_features'].shape}")


## 9. Loss function — Binary Cross Entropy with Logits

We use `BCEWithLogitsLoss` (Binary Cross Entropy) rather than the more common
`CrossEntropyLoss`. Here is why.

### CrossEntropyLoss assumes one correct answer

CrossEntropyLoss expects exactly one correct class per sample. It maximizes
the probability of that one class and minimizes all others.

### BCEWithLogitsLoss handles soft labels

Our VQA labels are soft score vectors: multiple answers can be partially
correct. BCEWithLogitsLoss treats each class independently as a binary
prediction task. A label of 0.67 for "yellow" means "yellow is 67% right"
— and the loss rewards the model for outputting a high score for yellow,
even if it's not 1.0.

### VQA accuracy metric

The official VQA accuracy is: `min(human_agreements / 3, 1.0)`.
We use this as our evaluation metric (not loss). A prediction is correct
if the predicted top answer matches any of the human answers, weighted by
agreement. We implement this below.


In [ ]:
def vqa_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """
    Soft-label Binary Cross Entropy loss for VQA.
    
    Args:
        logits: [B, num_classes] raw scores (not softmaxed)
        labels: [B, num_classes] soft scores in [0, 1]
    Returns:
        scalar loss
    """
    return F.binary_cross_entropy_with_logits(logits, labels)


def vqa_accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    """
    Official VQA accuracy: score of the predicted top answer.
    
    For each sample, we take the argmax of logits (the model's top prediction)
    and look up its soft score in the label vector.
    Average over the batch.
    
    Args:
        logits: [B, num_classes]
        labels: [B, num_classes] soft scores
    Returns:
        mean accuracy in [0, 1]
    """
    # Predicted answer index for each sample
    pred_idx = logits.argmax(dim=1)             # [B]
    # Score of the predicted answer for each sample
    scores   = labels[torch.arange(len(labels)), pred_idx]  # [B]
    return scores.mean().item()


# ── Test loss and accuracy ────────────────────────────────────────────────────
labels = torch.stack([samples[i].label for i in range(4)]).to(DEVICE)
logits = fused_out["logits"]

loss = vqa_loss(logits, labels)
acc  = vqa_accuracy(logits, labels)

print(f"Loss (random init, untrained) : {loss.item():.4f}")
print(f"VQA accuracy (random init)    : {acc:.4f}")
print(f"  (Expected near 0.0 for random weights — this is correct)")
print(f"  (After training, this will be ~0.45-0.60 for frozen encoders)")


## 10. Visualize features with PCA

This is one of the most powerful diagnostic tools in deep learning:
projecting high-dimensional feature vectors down to 2D using PCA and
plotting them. It lets us see whether the encoder is producing
meaningful, separable representations.

### What we look for

- **Clustering**: samples with similar answers should cluster together
- **Separation**: different answer types should occupy different regions
- **Corruption effect**: corrupted features should drift away from their
  clean counterparts — this drift is what the reliability estimator detects

### What is PCA?

Principal Component Analysis finds the two directions of maximum variance
in a high-dimensional space and projects all points onto those directions.
It's the standard first tool for visualizing embedding spaces.
A [2048]-dimensional feature vector becomes a 2D point you can plot.


In [ ]:
def extract_features_for_pca(model, samples, corruption_module=None,
                              n=64, model_type="fusion"):
    """
    Run samples through a model and collect features for PCA visualization.
    
    Args:
        model            : one of our baseline models
        samples          : list of MultimodalSample
        corruption_module: if not None, corrupt samples before encoding
        n                : number of samples to use
        model_type       : 'image', 'text', or 'fusion'
    Returns:
        features   : np.array [n, feature_dim]
        top_answers: list of top answer strings for coloring
    """
    all_features = []
    all_answers  = []
    
    model.eval()
    with torch.no_grad():
        for i in range(min(n, len(samples))):
            s = samples[i]
            if corruption_module is not None:
                s = corruption_module(s)
            
            batch = {
                "image"          : s.image.unsqueeze(0),
                "text_ids"       : s.text_ids.unsqueeze(0),
                "attention_mask" : s.attention_mask.unsqueeze(0),
                "label"          : s.label.unsqueeze(0),
            }
            
            out = model(batch)
            
            # Get the right feature vector for this model type
            if model_type == "image":
                feat = out["features"]
            elif model_type == "text":
                feat = out["features"]
            else:  # fusion
                feat = out["fused_features"]
            
            all_features.append(feat.squeeze(0).cpu().numpy())
            
            # Top answer for coloring the plot
            top_idx = s.label.argmax().item()
            all_answers.append(idx2answer.get(top_idx, "other"))
    
    return np.array(all_features), all_answers


def plot_pca_features(features_clean, features_corrupted,
                      answers, title="PCA Feature Space"):
    """
    Plot PCA of clean vs corrupted features side by side.
    Each point is one sample, colored by its top answer.
    Arrows show how corruption shifts each sample's features.
    """
    # Fit PCA on clean features
    pca = PCA(n_components=2)
    clean_2d = pca.fit_transform(features_clean)
    corr_2d  = pca.transform(features_corrupted)
    
    # Color by answer (top 8 answers get distinct colors, rest = gray)
    from collections import Counter
    answer_counts = Counter(answers)
    top_answers   = [a for a, _ in answer_counts.most_common(8)]
    colors_map    = {a: plt.cm.tab10(i) for i, a in enumerate(top_answers)}
    point_colors  = [colors_map.get(a, (0.7, 0.7, 0.7, 0.5)) for a in answers]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # ── Plot 1: Clean features ────────────────────────────────────────────────
    axes[0].scatter(clean_2d[:, 0], clean_2d[:, 1],
                    c=point_colors, s=60, alpha=0.8, edgecolors="white", lw=0.5)
    axes[0].set_title("Clean features", fontsize=12)
    axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
    axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
    
    # ── Plot 2: Corrupted features ────────────────────────────────────────────
    axes[1].scatter(corr_2d[:, 0], corr_2d[:, 1],
                    c=point_colors, s=60, alpha=0.8, edgecolors="white",
                    lw=0.5, marker="^")
    axes[1].set_title("Corrupted features", fontsize=12)
    axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
    
    # ── Plot 3: Overlay with drift arrows ─────────────────────────────────────
    axes[2].scatter(clean_2d[:, 0], clean_2d[:, 1],
                    c=point_colors, s=40, alpha=0.5, label="Clean", marker="o")
    axes[2].scatter(corr_2d[:, 0], corr_2d[:, 1],
                    c=point_colors, s=40, alpha=0.5, label="Corrupted", marker="^")
    
    # Draw arrows showing how each sample drifts under corruption
    for i in range(len(clean_2d)):
        axes[2].annotate("",
            xy=(corr_2d[i, 0], corr_2d[i, 1]),
            xytext=(clean_2d[i, 0], clean_2d[i, 1]),
            arrowprops=dict(arrowstyle="->", color="gray", alpha=0.3, lw=0.8)
        )
    
    axes[2].set_title("Feature drift under corruption", fontsize=12)
    axes[2].set_xlabel(f"PC1")
    axes[2].legend(fontsize=9)
    
    # ── Legend for answer colors ──────────────────────────────────────────────
    patches = [mpatches.Patch(color=colors_map[a], label=a)
               for a in top_answers]
    fig.legend(handles=patches, loc="lower center", ncol=4,
               fontsize=8, title="Top answer", bbox_to_anchor=(0.5, -0.05))
    
    # Compute mean drift distance
    drift = np.linalg.norm(corr_2d - clean_2d, axis=1).mean()
    fig.suptitle(f"{title}
Mean feature drift under corruption: {drift:.3f}",
                 fontsize=13, y=1.02)
    plt.tight_layout()
    return fig, pca, clean_2d, corr_2d


print("PCA visualization functions defined.")


In [ ]:
# ── Extract and plot features for each model ──────────────────────────────────
print("Extracting features for PCA visualization...")
print("(This runs 64 samples through each model — may take 1-2 minutes on CPU)")

# Corruption module at severity 3 (moderate) for visualization
vis_corruption = CorruptionModule(
    p_corrupt_image=1.0,   # always corrupt image for this visualization
    p_corrupt_text=0.0,
    p_missing_image=0.0,
    p_missing_text=0.0,
    severity=3,
    image_corruptions=["gaussian_noise"],
)

# ── Image model features ───────────────────────────────────────────────────────
print("Image model...")
img_feat_clean, img_answers = extract_features_for_pca(
    img_model, samples, corruption_module=None, model_type="image")
img_feat_corr, _            = extract_features_for_pca(
    img_model, samples, corruption_module=vis_corruption, model_type="image")

fig, _, _, _ = plot_pca_features(
    img_feat_clean, img_feat_corr, img_answers,
    title="Image encoder (ResNet-50) — Gaussian noise severity 3"
)
plt.savefig("pca_image_features.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: pca_image_features.png")


In [ ]:
# ── Text model features ───────────────────────────────────────────────────────
print("Text model (image corruption does not affect text features)...")
txt_feat_clean, txt_answers = extract_features_for_pca(
    txt_model, samples, corruption_module=None, model_type="text")
txt_feat_corr, _            = extract_features_for_pca(
    txt_model, samples, corruption_module=vis_corruption, model_type="text")

fig, _, _, _ = plot_pca_features(
    txt_feat_clean, txt_feat_corr, txt_answers,
    title="Text encoder (BERT) — image corruption does not affect text"
)
plt.savefig("pca_text_features.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: pca_text_features.png")
print("Note: text features should NOT drift when only the image is corrupted.")
print("This confirms the text encoder is independent of image corruption.")


In [ ]:
# ── Fusion model features ─────────────────────────────────────────────────────
print("Fusion model...")
fus_feat_clean, fus_answers = extract_features_for_pca(
    fusion_model, samples, corruption_module=None, model_type="fusion")
fus_feat_corr, _            = extract_features_for_pca(
    fusion_model, samples, corruption_module=vis_corruption, model_type="fusion")

fig, _, _, _ = plot_pca_features(
    fus_feat_clean, fus_feat_corr, fus_answers,
    title="Naive fusion features — concatenated image+text"
)
plt.savefig("pca_fusion_features.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: pca_fusion_features.png")
print()
print("Key observation: the fusion features drift MORE than text-only features")
print("because the corrupted image drags the concatenated representation away")
print("from its clean position. This is exactly the problem ARAF fixes.")


## 11. Corruption robustness test — the key diagnostic

This is the most important experiment in this notebook. We run all three
baseline models on the same samples under increasing corruption severity
and measure VQA accuracy at each level.

**What we expect to see:**
- All models degrade as severity increases (expected)
- Image model degrades fastest under image corruption (it has no text fallback)
- Text model stays flat under image corruption (it doesn't use the image)
- Naive fusion degrades more than text-only (corrupted image hurts it)

**Why this matters for your paper:**
This figure directly motivates ARAF. It shows that naive fusion is worse than
text-only under image corruption — proof that blindly combining modalities
can be harmful. ARAF's reliability weighting fixes this.


In [ ]:
def evaluate_model_under_corruption(
    model, samples, corruption_type: str,
    severities: List[int] = [0, 1, 2, 3, 4, 5],
    model_type: str = "fusion",
) -> Dict[int, float]:
    """
    Evaluate model accuracy at each corruption severity level.
    Severity 0 = clean (no corruption).
    
    Returns dict: {severity: accuracy}
    """
    results = {}
    model.eval()
    
    is_image_corruption = corruption_type in [
        "gaussian_noise", "motion_blur", "occlusion"]
    
    for sev in severities:
        if sev == 0:
            corr_module = None  # clean
        else:
            corr_module = CorruptionModule(
                p_corrupt_image = 1.0 if is_image_corruption else 0.0,
                p_corrupt_text  = 0.0 if is_image_corruption else 1.0,
                p_missing_image = 0.0,
                p_missing_text  = 0.0,
                severity        = sev,
                image_corruptions=[corruption_type] if is_image_corruption else None,
                text_corruptions =[corruption_type] if not is_image_corruption else None,
            )
        
        accs = []
        with torch.no_grad():
            for s in samples:
                s_c = corr_module(s) if corr_module else s
                batch = {
                    "image"          : s_c.image.unsqueeze(0),
                    "text_ids"       : s_c.text_ids.unsqueeze(0),
                    "attention_mask" : s_c.attention_mask.unsqueeze(0),
                    "label"          : s_c.label.unsqueeze(0),
                }
                out = model(batch)
                acc = vqa_accuracy(out["logits"], batch["label"].to(DEVICE))
                accs.append(acc)
        
        results[sev] = np.mean(accs)
    
    return results


def plot_robustness_curves(results_by_model: Dict, corruption_type: str):
    """
    Plot accuracy vs severity curves for multiple models.
    This is a standard figure in robustness papers.
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    
    colors = {"UnimodalImage": "#7F77DD",
              "UnimodalText" : "#1D9E75",
              "NaiveFusion"  : "#D85A30"}
    
    for model_name, results in results_by_model.items():
        severities = sorted(results.keys())
        accs       = [results[s] for s in severities]
        ax.plot(severities, accs, marker="o", linewidth=2,
                color=colors.get(model_name, "gray"),
                label=model_name, markersize=6)
        # Shade the drop from clean (severity 0) to severity 5
        ax.fill_between(severities, accs,
                        alpha=0.08, color=colors.get(model_name, "gray"))
    
    ax.set_xlabel("Corruption severity (0 = clean)", fontsize=11)
    ax.set_ylabel("VQA accuracy", fontsize=11)
    ax.set_title(f"Robustness under '{corruption_type}' corruption
"
                 f"(random weights — shape of curves matters, not absolute values)",
                 fontsize=11)
    ax.legend(fontsize=10)
    ax.set_xticks([0, 1, 2, 3, 4, 5])
    ax.set_xlim(-0.1, 5.1)
    ax.grid(alpha=0.3)
    
    # Annotate the key finding
    ax.annotate("Naive fusion should drop
more than text-only
(motivation for ARAF)",
                xy=(4, 0.02), fontsize=8, color="gray",
                ha="center", style="italic")
    
    plt.tight_layout()
    return fig


# ── Run evaluation ────────────────────────────────────────────────────────────
print("Running robustness evaluation across corruption severities...")
print("Testing on gaussian_noise (image corruption)...")
print("(Using random-weight models -- curves show relative behavior, not final accuracy)")

models_to_eval = {
    "UnimodalImage" : (img_model, "image"),
    "UnimodalText"  : (txt_model, "text"),
    "NaiveFusion"   : (fusion_model, "fusion"),
}

robustness_results = {}
for model_name, (model, mtype) in models_to_eval.items():
    print(f"  Evaluating {model_name}...")
    robustness_results[model_name] = evaluate_model_under_corruption(
        model, samples[:32],  # use 32 samples for speed
        corruption_type="gaussian_noise",
        model_type=mtype,
    )
    accs = robustness_results[model_name]
    print(f"    Severity 0 (clean): {accs[0]:.4f}")
    print(f"    Severity 3 (mid)  : {accs[3]:.4f}")
    print(f"    Severity 5 (max)  : {accs[5]:.4f}")

fig = plot_robustness_curves(robustness_results, "gaussian_noise")
plt.savefig("robustness_curves.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: robustness_curves.png")


## 12. Architecture comparison visualization

A clean diagram showing all three baselines and where ARAF will differ.
This is suitable for a paper figure or presentation slide.


In [ ]:
def draw_architecture_comparison():
    """
    Draw a clean comparison of all three baseline architectures
    and preview where ARAF differs.
    """
    fig, axes = plt.subplots(1, 4, figsize=(20, 6))
    titles = [
        "Unimodal Image",
        "Unimodal Text",
        "Naive Fusion",
        "ARAF (next notebook)",
    ]
    colors = ["#EEEDFE", "#E1F5EE", "#FAECE7", "#E6F1FB"]
    border = ["#7F77DD", "#1D9E75", "#D85A30", "#378ADD"]

    specs = [
        # (label, x, y, w, h, color)
        # Unimodal Image
        [("Image", 0.5, 0.85, 0.7, 0.12, "#D3D1C7"),
         ("ResNet-50", 0.5, 0.65, 0.7, 0.12, "#AFA9EC"),
         ("[2048]", 0.5, 0.45, 0.5, 0.10, "#7F77DD"),
         ("MLP", 0.5, 0.27, 0.5, 0.10, "#AFA9EC"),
         ("Logits", 0.5, 0.10, 0.5, 0.10, "#26215C")],
        # Unimodal Text
        [("Text", 0.5, 0.85, 0.7, 0.12, "#D3D1C7"),
         ("BERT", 0.5, 0.65, 0.7, 0.12, "#9FE1CB"),
         ("[768]", 0.5, 0.45, 0.5, 0.10, "#1D9E75"),
         ("MLP", 0.5, 0.27, 0.5, 0.10, "#9FE1CB"),
         ("Logits", 0.5, 0.10, 0.5, 0.10, "#04342C")],
        # Naive Fusion
        [("Image", 0.3, 0.85, 0.4, 0.10, "#D3D1C7"),
         ("Text", 0.7, 0.85, 0.4, 0.10, "#D3D1C7"),
         ("ResNet", 0.3, 0.70, 0.4, 0.10, "#AFA9EC"),
         ("BERT", 0.7, 0.70, 0.4, 0.10, "#9FE1CB"),
         ("Concat [2816]", 0.5, 0.50, 0.7, 0.10, "#F5C4B3"),
         ("MLP", 0.5, 0.30, 0.5, 0.10, "#F0997B"),
         ("Logits", 0.5, 0.10, 0.5, 0.10, "#4A1B0C")],
        # ARAF preview
        [("Image", 0.3, 0.88, 0.4, 0.09, "#D3D1C7"),
         ("Text", 0.7, 0.88, 0.4, 0.09, "#D3D1C7"),
         ("ResNet", 0.3, 0.74, 0.4, 0.09, "#AFA9EC"),
         ("BERT", 0.7, 0.74, 0.4, 0.09, "#9FE1CB"),
         ("Reliability
Estimator", 0.5, 0.57, 0.6, 0.10, "#FAC775"),
         ("Weighted
Fusion", 0.5, 0.40, 0.6, 0.10, "#B5D4F4"),
         ("MLP", 0.5, 0.24, 0.5, 0.09, "#85B7EB"),
         ("Logits", 0.5, 0.09, 0.5, 0.09, "#042C53")],
    ]

    for ax, title, bg, bc, spec in zip(axes, titles, colors, border, specs):
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_facecolor(bg)
        for spine in ax.spines.values():
            spine.set_edgecolor(bc); spine.set_linewidth(2)
        ax.set_title(title, fontsize=12, fontweight="bold", color=bc, pad=8)
        ax.axis("off")

        for (lbl, x, y, w, h, c) in spec:
            rect = mpatches.FancyBboxPatch(
                (x - w/2, y - h/2), w, h,
                boxstyle="round,pad=0.01",
                facecolor=c, edgecolor="white", linewidth=1.5,
                transform=ax.transAxes, clip_on=False,
            )
            ax.add_patch(rect)
            ax.text(x, y, lbl, ha="center", va="center",
                    fontsize=8.5, fontweight="bold",
                    transform=ax.transAxes, color="#1a1a1a")

    # ARAF label
    axes[3].text(0.5, -0.04, "Key: reliability estimator
weights each modality",
                 ha="center", va="top", fontsize=8, color="#185FA5",
                 transform=axes[3].transAxes, style="italic")

    fig.suptitle("Baseline architectures vs ARAF", fontsize=14, y=1.02,
                 fontweight="bold")
    plt.tight_layout()
    plt.savefig("architecture_comparison.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: architecture_comparison.png  (great for PPT/paper figures)")

draw_architecture_comparison()


## 13. Save baseline models as a .py file

We save the clean model definitions to models/baselines.py.
Notebook 04 (ARAF) and Notebook 05 (training) will import from here.


In [ ]:
import os
os.makedirs('models', exist_ok=True)
with open('models/__init__.py', 'w', encoding='utf-8') as f:
    f.write('')

baselines_lines = [
    'from typing import Optional, Dict',
    'import torch',
    'import torch.nn as nn',
    'import torch.nn.functional as F',
    'from torchvision import models',
    'from transformers import BertModel',
    '',
    'class ImageEncoder(nn.Module):',
    '    def __init__(self, frozen=True, output_dim=None):',
    '        super().__init__()',
    '        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)',
    '        self.encoder = nn.Sequential(*list(resnet.children())[:-1])',
    '        self.feature_dim = 2048',
    '        self.projection = None',
    '        if output_dim is not None:',
    '            self.projection = nn.Sequential(nn.Linear(2048, output_dim), nn.LayerNorm(output_dim), nn.ReLU())',
    '            self.feature_dim = output_dim',
    '        if frozen:',
    '            for p in self.encoder.parameters(): p.requires_grad = False',
    '    def forward(self, images):',
    '        f = self.encoder(images).flatten(start_dim=1)',
    '        return self.projection(f) if self.projection else f',
    '',
    'class TextEncoder(nn.Module):',
    '    def __init__(self, frozen=True, output_dim=None):',
    '        super().__init__()',
    '        self.bert = BertModel.from_pretrained("bert-base-uncased")',
    '        self.feature_dim = 768',
    '        self.projection = None',
    '        if output_dim is not None:',
    '            self.projection = nn.Sequential(nn.Linear(768, output_dim), nn.LayerNorm(output_dim), nn.ReLU())',
    '            self.feature_dim = output_dim',
    '        if frozen:',
    '            for p in self.bert.parameters(): p.requires_grad = False',
    '    def forward(self, text_ids, attention_mask):',
    '        out = self.bert(input_ids=text_ids, attention_mask=attention_mask)',
    '        cls = out.last_hidden_state[:, 0, :]',
    '        return self.projection(cls) if self.projection else cls',
    '',
    'class ClassificationHead(nn.Module):',
    '    def __init__(self, input_dim, hidden_dim=1024, num_classes=3129, dropout=0.3):',
    '        super().__init__()',
    '        self.classifier = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes))',
    '    def forward(self, x): return self.classifier(x)',
    '',
    'class UnimodalImageModel(nn.Module):',
    '    def __init__(self, num_classes=3129, frozen_encoder=True):',
    '        super().__init__()',
    '        self.encoder = ImageEncoder(frozen=frozen_encoder)',
    '        self.classifier = ClassificationHead(self.encoder.feature_dim, num_classes=num_classes)',
    '        self.name = "UnimodalImage"',
    '    def forward(self, batch, device="cpu"):',
    '        f = self.encoder(batch["image"].to(device))',
    '        return {"logits": self.classifier(f), "features": f}',
    '',
    'class UnimodalTextModel(nn.Module):',
    '    def __init__(self, num_classes=3129, frozen_encoder=True):',
    '        super().__init__()',
    '        self.encoder = TextEncoder(frozen=frozen_encoder)',
    '        self.classifier = ClassificationHead(self.encoder.feature_dim, num_classes=num_classes)',
    '        self.name = "UnimodalText"',
    '    def forward(self, batch, device="cpu"):',
    '        f = self.encoder(batch["text_ids"].to(device), batch["attention_mask"].to(device))',
    '        return {"logits": self.classifier(f), "features": f}',
    '',
    'class NaiveFusionModel(nn.Module):',
    '    def __init__(self, num_classes=3129, frozen_encoders=True):',
    '        super().__init__()',
    '        self.image_encoder = ImageEncoder(frozen=frozen_encoders)',
    '        self.text_encoder = TextEncoder(frozen=frozen_encoders)',
    '        fusion_dim = self.image_encoder.feature_dim + self.text_encoder.feature_dim',
    '        self.classifier = ClassificationHead(fusion_dim, num_classes=num_classes)',
    '        self.name = "NaiveFusion"',
    '    def forward(self, batch, device="cpu"):',
    '        img_f = self.image_encoder(batch["image"].to(device))',
    '        txt_f = self.text_encoder(batch["text_ids"].to(device), batch["attention_mask"].to(device))',
    '        fused = torch.cat([img_f, txt_f], dim=1)',
    '        return {"logits": self.classifier(fused), "image_features": img_f, "text_features": txt_f, "fused_features": fused}',
    '',
    'def vqa_loss(logits, labels): return F.binary_cross_entropy_with_logits(logits, labels)',
    '',
    'def vqa_accuracy(logits, labels):',
    '    pred_idx = logits.argmax(dim=1)',
    '    scores = labels[torch.arange(len(labels)), pred_idx]',
    '    return scores.mean().item()',
]

with open('models/baselines.py', 'w', encoding='utf-8') as f:
    f.write('\n'.join(baselines_lines))

print('Saved: models/baselines.py')

import importlib.util
spec = importlib.util.spec_from_file_location('baselines', 'models/baselines.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
print('Import verified.')


## 14. Summary and what's next

### What we built

| Component | File | Purpose |
|---|---|---|
| `ImageEncoder` | `models/baselines.py` | ResNet-50 feature extractor |
| `TextEncoder` | `models/baselines.py` | BERT [CLS] feature extractor |
| `ClassificationHead` | `models/baselines.py` | MLP: features → class logits |
| `UnimodalImageModel` | `models/baselines.py` | Image-only baseline |
| `UnimodalTextModel` | `models/baselines.py` | Text-only baseline |
| `NaiveFusionModel` | `models/baselines.py` | Concat fusion baseline |
| `vqa_loss` | `models/baselines.py` | Soft-label BCE loss |
| `vqa_accuracy` | `models/baselines.py` | Official VQA scoring |

### What the visualizations showed

- **PCA plots**: Features form structure even with random weights.
  Under corruption, image features drift — this drift is what the reliability
  estimator in ARAF learns to detect.
- **Robustness curves**: All models degrade under corruption. The key finding
  (with trained models) will be that naive fusion degrades more than text-only
  under image corruption — proving that blind fusion is harmful.
- **Architecture diagram**: Ready for your PPT. The ARAF column previews
  exactly where the reliability estimator sits in the pipeline.

### Your project folder now looks like

```
your_project/
├── answer_vocab.json
├── config.json
├── corruption/
│   ├── __init__.py
│   └── corruption_module.py
├── models/
│   ├── __init__.py
│   └── baselines.py          <- added this notebook
├── *.png                     <- all visualizations
```

### What Notebook 04 covers

**The ARAF model** — the core contribution.

We add two new components on top of what we built today:

1. **Reliability Estimator**: a small network that looks at each modality's
   features and outputs a scalar confidence score (0 = unreliable, 1 = reliable).
   It learns that zero-filled images are unreliable, and heavily masked text
   is unreliable.

2. **Adaptive Fusion**: instead of concatenating features equally, we use the
   reliability scores as weights. If image reliability = 0.1 and text
   reliability = 0.9, the fusion is dominated by text — exactly what we want
   when the image is corrupted.
